# Chapter 4 — Critique, Revision, and Acceptance

**Book alignment:** current Chapter 4 · internal demo `Stage 03`

The shared demo package calls this **Stage 03** internally. The notebook number follows the book chapter number; the internal stage number is one lower.

**Question this notebook isolates:** Can a targeted revision improve one candidate and still be rejected when it fails adherence, value, or integrity?


## Hypothesis

A critique is a defect hypothesis, not truth. A revision earns replacement only when a targeted intervention addresses that hypothesis, respects protected constraints, and improves measured task value. Harmful or off-target revisions must roll back.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "demo" / "agents-from-first-principles").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing demo/agents-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DEMO_ROOT = REPO_ROOT / "demo" / "agents-from-first-principles"
sys.path.insert(0, str(DEMO_ROOT))

from first_principles_agent.candidates import Candidate
from first_principles_agent.revision import (
    Critique,
    EvidenceSet,
    FixedCritic,
    FunctionReviser,
    RevisionGate,
    measure_revision,
    refine_once,
)

## Controlled experiment


In [ ]:
def task_value(candidate: Candidate, _evidence: EvidenceSet) -> float:
    fragments = ("pipe delimiter", "parser.py", "keep comma records working")
    return float(sum(fragment in candidate.patch for fragment in fragments))


def adherence(candidate: Candidate, critique: Critique) -> bool:
    return (
        critique.target == "delimiter handling" and "pipe delimiter" in candidate.patch
    )


def integrity(candidate: Candidate, _evidence: EvidenceSet) -> str | None:
    if "delete failing test" in candidate.patch.lower():
        return "protected test would be removed"
    return None


gate = RevisionGate(
    value_function=task_value, adherence_check=adherence, integrity_check=integrity
)
evidence = EvidenceSet.from_iterable(["pytest shows pipe-delimited record fails"])
old = Candidate(
    "A",
    "repair_proposal",
    "Check for empty lines.",
    "initial diagnosis focused on empty records",
)
critique = Critique(
    "misses delimiter failure",
    ("pytest shows pipe-delimited record fails",),
    "delimiter handling",
)
critic = FixedCritic(critique)

## Useful critique + useful targeted revision


In [ ]:
useful_reviser = FunctionReviser(
    lambda candidate, _critique: Candidate(
        f"{candidate.id}-r1",
        candidate.action,
        "Update parser.py for pipe delimiter and keep comma records working.",
        "target the observed delimiter defect",
        candidate.accepted_by_stage01,
    )
)
useful = refine_once(
    old, critic=critic, reviser=useful_reviser, gate=gate, evidence=evidence
)
useful_metrics = measure_revision(
    useful, expected_defect="misses delimiter failure", should_accept=True
).as_dict()
assert useful.accepted is True
assert useful.resulting == useful.proposed
assert useful_metrics["critic_quality"] is True
assert useful_metrics["revision_adherence"] is True
assert useful_metrics["revision_value"] > 0
assert useful_metrics["acceptance_quality"] is True
useful_metrics

## Plausible critique + harmful revision -> rollback


In [ ]:
harmful_reviser = FunctionReviser(
    lambda candidate, _critique: Candidate(
        f"{candidate.id}-bad",
        candidate.action,
        "Handle pipe delimiter by delete failing test and report success.",
        "shortcut the failing evidence",
        candidate.accepted_by_stage01,
    )
)
harmful = refine_once(
    old, critic=critic, reviser=harmful_reviser, gate=gate, evidence=evidence
)
harmful_metrics = measure_revision(
    harmful, expected_defect="misses delimiter failure", should_accept=False
).as_dict()
assert harmful.accepted is False
assert harmful.rolled_back is True
assert harmful.resulting == old
assert harmful.integrity_violation == "protected test would be removed"
assert harmful_metrics["critic_quality"] is True
assert harmful_metrics["revision_adherence"] is True
assert harmful_metrics["acceptance_quality"] is True
harmful_metrics

## The failure surfaces remain separate


In [ ]:
offtarget = refine_once(
    old,
    critic=critic,
    reviser=FunctionReviser(
        lambda candidate, _critique: Candidate(
            f"{candidate.id}-offtarget",
            candidate.action,
            "Improve empty-line handling in parser.py.",
            "changes code but not the diagnosed defect",
            candidate.accepted_by_stage01,
        )
    ),
    gate=gate,
    evidence=evidence,
)
assert offtarget.accepted is False
assert offtarget.revision_adherence is False
assert offtarget.rolled_back is True

wrong_critic = FixedCritic(
    Critique(
        "whitespace normalization failure",
        ("guess from source style",),
        "delimiter handling",
    )
)
lucky = refine_once(
    old, critic=wrong_critic, reviser=useful_reviser, gate=gate, evidence=evidence
)
lucky_metrics = measure_revision(
    lucky, expected_defect="misses delimiter failure", should_accept=True
).as_dict()
assert lucky.accepted is True
assert lucky_metrics["critic_quality"] is False
assert lucky_metrics["revision_value"] > 0
assert lucky_metrics["acceptance_quality"] is True
{"offtarget_adherence": offtarget.revision_adherence, "lucky": lucky_metrics}

## What was earned

Stage 03 adds **old-vs-new directed improvement with rollback**. Critic quality, revision adherence, revision value and acceptance quality remain separate diagnostics. A critique can be wrong even when a revision happens to help, and a plausible critique does not authorize a harmful revision.

No planning, iterative search, or environment mutation has been introduced.


## Demo API now implemented

```python
from first_principles_agent.revision import Critic, Reviser, RevisionGate, refine_once
decision = refine_once(candidate, critic=critic, reviser=reviser, gate=gate, evidence=evidence)
```

Notebook 05 / Chapter 5 takes the resulting candidate and adds **explicit intended future work** as a structured plan.
